# Самостоятельная работа 04. Полиномиальная регрессия, регуляризация и логистическая регрессия

Цель: по кривым обучения отличить недообучение от переобучения, обучить гребневую регрессию и лассо и объяснить разницу между ними, получить вероятности и границу решения логистической регрессии и обучить softmax-регрессию градиентным спуском с ранней остановкой.

Правила:

- защищать можно любое число заданий — от одного до всех;
- уйти с занятия раньше времени можно только после защиты хотя бы одного задания;
- баллы ставятся только за защищённые задания: базовое — 1 балл, продвинутое — 3 балла; выполненное, но не защищённое задание даёт 0;
- на защите нужно объяснить любую строку своего кода и при преподавателе изменить его;
- ИИ-помощником можно пользоваться точечно — вспомнить функцию, разобраться с ошибкой; код, который вы не можете объяснить, не засчитывается;
- перед защитой перезапустите ядро (Kernel → Restart) и выполните сверху вниз подготовку данных и защищаемые задания — их самопроверки должны пройти.

Материалы:

- лекция 05 «Полиномиальная регрессия, регуляризация и логистическая регрессия»;
- Орельен Жерон, «Прикладное машинное обучение с помощью Scikit-Learn и TensorFlow», гл. 4, стр. 172–197;
- блокнот практического занятия 05 «Обучение линейной регрессии: МНК и градиентный спуск» — градиентный спуск в векторной форме пригодится в задании 4.

| № | Что сделать | Тип | Баллы |
|---|---|---|---|
| 1 | Кривые обучения прямой и полинома 10-й степени | базовое | 1 |
| 2 | Гребневая регрессия в замкнутой форме и лассо | базовое | 1 |
| 3 | Вероятности и граница решения логистической регрессии | базовое | 1 |
| 4 | Softmax-регрессия градиентным спуском с ранней остановкой | продвинутое | 3 |

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn

sys.version, np.__version__, sklearn.__version__, matplotlib.__version__

## Подготовка данных

Ячейки ниже готовы — просто выполните их. Первая повторяет пример книги (стр. 172): точки квадратичной зависимости $y = 0{,}5x^2 + x + 2$ с гауссовым шумом (стандартное отклонение 1) на отрезке $[-3, 3]$. Точки уже перемешаны и разделены пополам:

- `x_train`, `y_train` — обучающий набор, 100 точек, `x_train` — матрица формы (100, 1);
- `x_val`, `y_val` — валидационный набор (англ. *validation set*), 100 точек.

In [ ]:
from sklearn.model_selection import train_test_split


def make_quadratic_data(n_points, seed=0):
    """Генерирует зашумлённые точки квадратичной зависимости, как в примере книги.

    Args:
        n_points: Число точек.
        seed: Зерно генератора случайных чисел.

    Returns:
        Матрица признака формы (n_points, 1) и вектор целевых значений.
    """
    rng = np.random.default_rng(seed)
    feature = 6 * rng.random((n_points, 1)) - 3
    targets = 0.5 * feature[:, 0] ** 2 + feature[:, 0] + 2 + rng.normal(size=n_points)
    return feature, targets


x_quad, y_quad = make_quadratic_data(n_points=200, seed=0)
x_train, x_val, y_train, y_val = train_test_split(x_quad, y_quad, test_size=0.5, random_state=42)

x_train.shape, x_val.shape

Вторая ячейка — «широкая» таблица для задания 2: 60 образцов и 30 признаков, но целевое значение зависит только от 5 из них, остальные — шум. Признаков много относительно образцов, и линейная модель без ограничений легко переобучается.

- `X_wide`, `y_wide` — матрица признаков формы (60, 30) и целевые значения; их среднее около 50;
- `true_weights` — истинные веса, по которым сгенерированы данные: у 25 шумовых признаков вес 0;
- `informative_features` — номера 5 полезных признаков.

In [ ]:
from sklearn.datasets import make_regression

X_wide, y_wide, true_weights = make_regression(
    n_samples=60,
    n_features=30,
    n_informative=5,
    noise=5.0,
    bias=50.0,
    coef=True,
    random_state=42,
)
informative_features = np.flatnonzero(true_weights)

X_wide.shape, informative_features

Третья ячейка загружает набор «Ирисы» (англ. *Iris*) для заданий 3 и 4: 150 цветков трёх видов — щетинистый (0), разноцветный (1) и виргинский (2), у каждого 4 признака в сантиметрах: длина и ширина чашелистика, длина и ширина лепестка.

- `petal_width` — ширина лепестка, матрица формы (150, 1); `is_virginica` — 1, если цветок виргинский, иначе 0;
- `X_iris_train`, `y_iris_train` — 105 цветков для обучения, все 4 признака и метки видов 0, 1, 2;
- `X_iris_val`, `y_iris_val` — 45 цветков для валидации; доли видов в обоих наборах одинаковые.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
petal_width = iris['data'][:, 3:]  # Срез 3: сохраняет матрицу (150, 1), которую ждёт scikit-learn
is_virginica = (iris['target'] == 2).astype(int)
X_iris_train, X_iris_val, y_iris_train, y_iris_val = train_test_split(
    iris['data'],
    iris['target'],
    test_size=0.3,
    stratify=iris['target'],
    random_state=42,
)

petal_width.shape, is_virginica.sum(), X_iris_train.shape, X_iris_val.shape

## Задание 1 (базовое, 1 балл)

Истинная зависимость обычно неизвестна, и по одной ошибке на валидации трудно понять, что не так с моделью: она слишком проста или слишком сложна. Кривые обучения (англ. *learning curves*) показывают ошибку на обучающем и на валидационном наборе в зависимости от размера обучающего набора — по их форме видно и недообучение, и переобучение. Постройте кривые для прямой и для полинома 10-й степени на квадратичных данных.

**Дано:** `x_train`, `y_train`, `x_val`, `y_val` из подготовки; `train_sizes` — размеры обучающего набора от 2 до 100, массив уже записан в заготовке.

**Получить:**

- функцию `learning_curve_rmse(model, x_train, y_train, x_val, y_val, train_sizes)`, которая возвращает кортеж из двух массивов NumPy длины `len(train_sizes)`: `train_rmse` — RMSE на тех обучающих точках, на которых модель обучалась, и `val_rmse` — RMSE на всём валидационном наборе;
- `linear_model` — линейная регрессия на исходном признаке `x`;
- `poly10_model` — полиномиальная регрессия 10-й степени: конвейер из трёх шагов — добавить степени признака от 1 до 10, масштабировать их, обучить линейную регрессию;
- `train_rmse_linear`, `val_rmse_linear` и `train_rmse_poly10`, `val_rmse_poly10` — кривые обеих моделей.

**Как действовать:**

1. Сначала запишите предсказание в ячейку ниже.
2. В функции для каждого размера `size` из `train_sizes` обучите модель на первых `size` точках обучающего набора (точки уже перемешаны) и посчитайте корень из среднеквадратичной ошибки (англ. *RMSE*) на этих же точках и на всём валидационном наборе.
3. Соберите обе модели. Для полиномиальной: столбец единиц добавлять не нужно — свободный член добавит линейная регрессия.
4. Вызовите функцию для обеих моделей и выполните готовую ячейку с графиком после самопроверки.

Функцию `learning_curve` из scikit-learn здесь не используйте: задание про то, как устроена кривая.

**Что почитать:**

- лекция 05, слайды «Степени признака как новые признаки», «Кривые обучения прямой», «Кривые обучения полинома 10-й степени» и «Как читать кривые обучения»; книга, стр. 172–178;
- [Переобучение](https://ru.wikipedia.org/wiki/Переобучение) и [Дилемма смещения–дисперсии](https://ru.wikipedia.org/wiki/Дилемма_смещения–дисперсии) — Википедия;
- [Learning curve (machine learning)](https://en.wikipedia.org/wiki/Learning_curve_%28machine_learning%29) — Википедия на английском;
- документация: [`PolynomialFeatures`](https://scikit-learn.org/1.3/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [`make_pipeline`](https://scikit-learn.org/1.3/modules/generated/sklearn.pipeline.make_pipeline.html), [`mean_squared_error`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.mean_squared_error.html) (параметр `squared`);
- руководство scikit-learn: [Validation curves and learning curves](https://scikit-learn.org/1.3/modules/learning_curve.html).

Предсказание: не запуская код, ответьте. (1) Чему равна обучающая RMSE прямой на двух точках? (2) Будет ли полином 10-й степени на 5–10 обучающих точках хорошо предсказывать валидационный набор? (3) Опустятся ли кривые прямой на 100 точках ниже 1 — стандартного отклонения шума?

*Ваш ответ:* …

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


def learning_curve_rmse(model, x_train, y_train, x_val, y_val, train_sizes):
    """Считает кривые обучения модели по RMSE.

    Args:
        model: Модель scikit-learn с методами fit и predict.
        x_train: Матрица признаков обучающего набора.
        y_train: Целевые значения обучающего набора.
        x_val: Матрица признаков валидационного набора.
        y_val: Целевые значения валидационного набора.
        train_sizes: Размеры обучающего набора, для которых считаются ошибки.

    Returns:
        Кортеж (train_rmse, val_rmse) — массивы NumPy длины len(train_sizes).
    """
    ...  # Ваш код


train_sizes = np.arange(2, len(x_train) + 1)
linear_model = ...  # Ваш код
poly10_model = ...  # Ваш код
train_rmse_linear, val_rmse_linear = ..., ...  # Ваш код
train_rmse_poly10, val_rmse_poly10 = ..., ...  # Ваш код

train_rmse_linear[-1], val_rmse_linear[-1], train_rmse_poly10[-1], val_rmse_poly10[-1]

In [ ]:
curves = [train_rmse_linear, val_rmse_linear, train_rmse_poly10, val_rmse_poly10]
assert all(np.shape(curve) == train_sizes.shape for curve in curves), 'Нужно по одной RMSE на каждый размер train_sizes'
assert train_rmse_linear[0] < 1e-6, 'Через две точки прямая проходит точно: обучающая RMSE на двух точках равна 0'
assert val_rmse_linear[-1] > 1.3, 'Прямая не описывает параболу: валидационная RMSE должна остаться выше шума'
assert abs(train_rmse_linear[-1] - val_rmse_linear[-1]) < 0.3, 'На 100 точках кривые прямой должны почти сойтись'
assert train_rmse_poly10[-1] < train_rmse_linear[-1], 'Полином 10-й степени должен ошибаться на обучении меньше прямой'
assert np.max(val_rmse_poly10[:10]) > 10 * val_rmse_poly10[-1], 'На малых наборах полином должен сильно переобучаться'
assert val_rmse_poly10[-1] < val_rmse_linear[-1], 'На 100 точках полином должен обобщать лучше прямой'

'Задание 1: самопроверка пройдена'

In [ ]:
curves_by_model = [
    ('Прямая', train_rmse_linear, val_rmse_linear),
    ('Полином 10-й степени', train_rmse_poly10, val_rmse_poly10),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for axis, (title, train_rmse, val_rmse) in zip(axes, curves_by_model):
    axis.plot(train_sizes, train_rmse, label='обучающий набор')
    axis.plot(train_sizes, val_rmse, label='валидационный набор')
    axis.set_title(title)
    axis.set_xlabel('Размер обучающего набора')
    axis.set_ylim(0, 3)  # Валидационная RMSE полинома на малых наборах уходит далеко за край
    axis.grid(alpha=0.3)

axes[0].set_ylabel('RMSE')
axes[0].legend()
plt.show()

По своему графику: (1) какая модель недообучается, а какая на малых наборах переобучается, и по каким признакам кривых вы это видите? (2) Поможет ли каждой из моделей ещё 1000 обучающих точек?

*Ваш ответ:* …

## Задание 2 (базовое, 1 балл)

Регуляризация ограничивает веса модели, чтобы ей было труднее переобучиться. Гребневая регрессия (англ. *ridge regression*) штрафует сумму квадратов весов и решается в замкнутой форме — как нормальное уравнение с поправкой. Лассо-регрессия (англ. *lasso regression*) штрафует сумму модулей весов и обнуляет веса бесполезных признаков. Решите гребневую регрессию сами, сверьтесь с scikit-learn и сравните её с лассо на «широкой» таблице.

**Дано:** `X_wide`, `y_wide` — 60 образцов, 30 признаков, из них полезны 5 (`informative_features`).

**Получить:**

- функцию `ridge_closed_form(features, targets, alpha)`, которая возвращает кортеж `(intercept, weights)` — свободный член и вектор весов длины `n`;
- `intercept_own`, `weights_own` — результат вашей функции на `X_wide`, `y_wide` при `alpha = 1.0`;
- `intercept_sklearn`, `weights_sklearn` — свободный член и веса класса `Ridge` с тем же `alpha`;
- `weights_lasso` — веса класса `Lasso` с `alpha = 1.0` на тех же данных;
- `zeros_ridge`, `zeros_lasso` — сколько весов вашей гребневой регрессии и лассо равны ровно нулю, целые числа.

**Как действовать:**

1. Сначала запишите предсказание в ячейку ниже.
2. Добавьте к матрице признаков столбец единиц для свободного члена, как в нормальном уравнении лекции 04.
3. Запишите решение в замкнутой форме по уравнению 4.9 (слайд «Решение в замкнутой форме», стр. 181). Обратите внимание на матрицу $\mathbf{A}$: свободный член штрафовать нельзя. Систему решайте функцией `np.linalg.solve`, а не обращением матрицы.
4. Обучите `Ridge` и `Lasso` и посчитайте нулевые веса.

Внутри `ridge_closed_form` классы scikit-learn не используйте: задание про формулу.

**Что почитать:**

- лекция 05, слайды «Зачем ограничивать веса», «Гребневая регрессия», «Решение в замкнутой форме», «Лассо-регрессия» и «Почему лассо обнуляет веса»; книга, стр. 179–184;
- [Гребневая регрессия](https://ru.wikipedia.org/wiki/Гребневая_регрессия) — Википедия; [Lasso (statistics)](https://en.wikipedia.org/wiki/Lasso_%28statistics%29) — Википедия на английском;
- документация: [`Ridge`](https://scikit-learn.org/1.3/modules/generated/sklearn.linear_model.Ridge.html) и [`Lasso`](https://scikit-learn.org/1.3/modules/generated/sklearn.linear_model.Lasso.html) — какую функцию стоимости минимизирует каждый класс; [`np.linalg.solve`](https://numpy.org/doc/1.24/reference/generated/numpy.linalg.solve.html), [`np.eye`](https://numpy.org/doc/1.24/reference/generated/numpy.eye.html), [`np.c_`](https://numpy.org/doc/1.24/reference/generated/numpy.c_.html);
- руководство scikit-learn: [Ridge regression](https://scikit-learn.org/1.3/modules/linear_model.html#ridge-regression-and-classification) и [Lasso](https://scikit-learn.org/1.3/modules/linear_model.html#lasso).

Предсказание: не запуская код, ответьте — сколько из 30 весов окажутся ровно нулём у гребневой регрессии и у лассо при `alpha = 1.0`?

*Ваш ответ:* …

In [ ]:
from sklearn.linear_model import Lasso, Ridge


def ridge_closed_form(features, targets, alpha):
    """Обучает гребневую регрессию решением в замкнутой форме.

    Args:
        features: Матрица признаков формы (m, n) без столбца единиц.
        targets: Вектор целевых значений длины m.
        alpha: Сила регуляризации.

    Returns:
        Кортеж (intercept, weights): свободный член и вектор весов длины n.
    """
    ...  # Ваш код


intercept_own, weights_own = ..., ...  # Ваша функция, alpha = 1.0
intercept_sklearn, weights_sklearn = ..., ...  # Ridge, alpha = 1.0
weights_lasso = ...  # Lasso, alpha = 1.0
zeros_ridge, zeros_lasso = ..., ...  # Ваш код

zeros_ridge, zeros_lasso

In [ ]:
toy_features = np.array([[0.0], [1.0], [2.0], [3.0]])
toy_intercept, toy_weights = ridge_closed_form(toy_features, 3 + 2 * toy_features[:, 0], alpha=0.0)
assert np.isclose(toy_intercept, 3) and np.allclose(toy_weights, [2]), 'При alpha = 0 для y = 3 + 2x нужно 3 и [2]'
heavy_intercept, heavy_weights = ridge_closed_form(X_wide, y_wide, alpha=1e9)
assert np.allclose(heavy_weights, 0, atol=1e-3), 'При огромном alpha все веса должны стать почти нулевыми'
assert np.isclose(heavy_intercept, y_wide.mean(), atol=0.1), 'При огромном alpha свободный член ≈ среднее y'
assert np.shape(weights_own) == np.shape(weights_lasso) == (30,), 'Нужно по одному весу на каждый из 30 признаков'
assert np.allclose(weights_own, weights_sklearn), 'Ваши веса не совпали с Ridge: проверьте матрицу A и alpha'
assert np.isclose(intercept_own, intercept_sklearn), 'Ваш свободный член не совпал с Ridge'
assert zeros_ridge == 0, 'Ни один вес гребневой регрессии не должен быть ровно нулём'
assert 15 <= zeros_lasso < 30, 'Лассо должно обнулить больше половины весов, но не все'

'Задание 2: самопроверка пройдена'

In [ ]:
feature_numbers = np.arange(len(true_weights))
fig, axis = plt.subplots(figsize=(11, 3.5))
axis.bar(feature_numbers - 0.27, true_weights, width=0.27, label='истинные веса')
axis.bar(feature_numbers, weights_own, width=0.27, label='гребневая регрессия')
axis.bar(feature_numbers + 0.27, weights_lasso, width=0.27, label='лассо')
axis.set_xlabel('Номер признака')
axis.set_ylabel('Вес')
axis.legend()
axis.grid(alpha=0.3)
plt.show()

np.flatnonzero(weights_lasso), informative_features

По своему графику и выводу: (1) совпадают ли признаки, которые оставило лассо, с полезными `informative_features`, и чем отличаются веса оставшихся «лишних» признаков? (2) Почему гребневая регрессия не обнуляет ни одного веса, хотя 25 признаков бесполезны?

*Ваш ответ:* …

## Задание 3 (базовое, 1 балл)

Логистическая регрессия (англ. *logistic regression*) считает ту же взвешенную сумму признаков, что и линейная, но пропускает её через логистическую функцию (англ. *logistic function*, *sigmoid*) и получает вероятность класса. Обучите классификатор «виргинский или нет» по ширине лепестка, как в книге, и получите его вероятности, границу решения и логарифмическую функцию потерь сами — из двух параметров модели.

**Дано:** `petal_width`, `is_virginica` из подготовки; сетка ширины лепестка `width_grid` от 0 до 3 см из 301 точки — уже записана в заготовке.

**Получить:**

- функцию `sigmoid(scores)` — логистическую функцию, работающую поэлементно на массивах;
- `virginica_clf` — `LogisticRegression(random_state=42)`, обученную на `petal_width` и `is_virginica`; остальные параметры — по умолчанию;
- `proba_own` — вероятности «виргинский» в точках `width_grid`, одномерный массив длины 301;
- `boundary_width` — ширина лепестка на границе решения (англ. *decision boundary*), число в сантиметрах;
- `log_loss_own` — логарифмическая функция потерь (англ. *log loss*) модели на всех 150 цветках по вашей формуле; `log_loss_sklearn` — то же функцией `log_loss`.

**Как действовать:**

1. Запишите логистическую функцию по уравнению 4.14 (слайд «Логистическая функция»).
2. Возьмите из обученной модели вес признака `coef_` и свободный член `intercept_` и посчитайте вероятности по уравнению 4.13.
3. Найдите границу решения из условия, при котором вероятность равна 0,5: какому значению взвешенной суммы оно соответствует? Решите получившееся уравнение относительно ширины лепестка.
4. Посчитайте вероятности на `petal_width` и логарифмическую функцию потерь по уравнению 4.17; для сверки — функция `log_loss` по меткам и `predict_proba`.

Для `proba_own` и `log_loss_own` методы `predict_proba` и `decision_function` не используйте, а `boundary_width` не ищите перебором сетки — задание про формулы модели.

**Что почитать:**

- лекция 05, слайды «Оценивание вероятностей», «Логистическая функция», «Цена ошибки одного образца», «Логарифмическая функция потерь» и «Один признак: ширина лепестка»; книга, стр. 187–192;
- [Логистическая регрессия](https://ru.wikipedia.org/wiki/Логистическая_регрессия) и [Сигмоида](https://ru.wikipedia.org/wiki/Сигмоида) — Википедия;
- документация: [`LogisticRegression`](https://scikit-learn.org/1.3/modules/generated/sklearn.linear_model.LogisticRegression.html) (атрибуты `coef_` и `intercept_`), [`log_loss`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.log_loss.html), [`np.exp`](https://numpy.org/doc/1.24/reference/generated/numpy.exp.html), [`np.log`](https://numpy.org/doc/1.24/reference/generated/numpy.log.html);
- руководство scikit-learn: [Logistic regression](https://scikit-learn.org/1.3/modules/linear_model.html#logistic-regression).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss


def sigmoid(scores):
    """Вычисляет логистическую функцию поэлементно.

    Args:
        scores: Число или массив NumPy.

    Returns:
        Значения логистической функции той же формы.
    """
    ...  # Ваш код


width_grid = np.linspace(0, 3, 301).reshape(-1, 1)
virginica_clf = ...  # Ваш код
proba_own = ...  # Вероятности «виргинский» на width_grid по вашей формуле
boundary_width = ...  # Ваш код
log_loss_own, log_loss_sklearn = ..., ...  # Ваш код

boundary_width, log_loss_own, log_loss_sklearn

In [ ]:
assert np.isclose(sigmoid(0), 0.5), 'Логистическая функция в нуле равна 0,5'
assert np.allclose(sigmoid(np.array([-50.0, 50.0])), [0, 1]), 'На краях логистическая функция близка к 0 и 1'
assert np.shape(proba_own) == (301,), 'proba_own — одномерный массив из 301 вероятности, по одной на точку width_grid'
assert np.allclose(proba_own, virginica_clf.predict_proba(width_grid)[:, 1]), 'Ваши вероятности не совпали с моделью'
assert 1.4 < boundary_width < 1.9, 'Граница решения на ирисах — между 1,4 и 1,9 см'
edge_widths = [[boundary_width - 0.01], [boundary_width + 0.01]]
assert list(virginica_clf.predict(edge_widths)) == [0, 1], 'Слева от границы — «не виргинский», справа — «виргинский»'
assert np.isclose(log_loss_own, log_loss_sklearn), 'Ваша логарифмическая функция потерь не совпала с log_loss'

'Задание 3: самопроверка пройдена'

In [ ]:
jitter = np.random.default_rng(0).uniform(-0.03, 0.03, size=len(is_virginica))  # Раздвигает совпавшие точки
fig, axis = plt.subplots(figsize=(9, 4))
axis.plot(width_grid[:, 0], proba_own, label='вероятность «виргинский»')
axis.scatter(petal_width[:, 0], is_virginica + jitter, s=12, alpha=0.5, color='gray', label='цветки (0 или 1)')
axis.axvline(boundary_width, color='red', linestyle='--', label='граница решения')
axis.axhline(0.5, color='red', linewidth=0.5)
axis.set_xlabel('Ширина лепестка, см')
axis.set_ylabel('Вероятность')
axis.legend()
axis.grid(alpha=0.3)
plt.show()

По своему графику и числам: (1) в каком диапазоне ширины лепестка модель «сомневается» — вероятность между 0,1 и 0,9 — и какой из двух параметров модели задаёт ширину этого диапазона? (2) Почему для границы решения не нужна сама логистическая функция — достаточно двух параметров?

*Ваш ответ:* …

## Задание 4 (продвинутое, 3 балла)

Softmax-регрессия (англ. *softmax regression*) обобщает логистическую регрессию на несколько классов: у каждого класса свой вектор параметров и своя оценка, функция softmax (англ. *softmax function*) переводит оценки в вероятности, а обучают модель, минимизируя перекрёстную энтропию (англ. *cross entropy*). Это упражнение 12 книги (стр. 198): обучите softmax-регрессию пакетным градиентным спуском с ранней остановкой (англ. *early stopping*) на NumPy, без scikit-learn, и сравните с `LogisticRegression`.

**Дано:** `X_iris_train`, `y_iris_train`, `X_iris_val`, `y_iris_val` — все 4 признака ирисов и метки видов 0, 1, 2; в заготовке записаны `learning_rate = 0.5`, `max_epochs = 5000` и `patience = 1000`.

**Получить** (первая ячейка):

- функцию `softmax(scores)` — матрица оценок формы (m, K) → матрица вероятностей той же формы; она не должна переполняться при больших оценках;
- функцию `cross_entropy(proba, targets_onehot)` — средняя по образцам перекрёстная энтропия по уравнению 4.22.

**Получить** (вторая ячейка):

- `train_losses`, `val_losses` — списки перекрёстной энтропии на обучающем и валидационном наборе: элемент с номером `k` — потеря параметров после `k` шагов спуска, элемент 0 — до первого шага;
- `best_epoch` — номер эпохи с наименьшей валидационной потерей, `theta_best` — матрица параметров формы (5, 3) в этой эпохе: строка 0 — свободные члены, по столбцу на класс;
- `val_accuracy_own` — доля правильных ответов вашей модели с `theta_best` на валидационном наборе;
- `val_accuracy_sklearn` — то же для `LogisticRegression(max_iter=1000, random_state=42)` на тех же данных.

**Как действовать:**

1. Функция softmax — уравнение 4.20. Чтобы `exp` не переполнялся, вычтите из оценок каждого образца их максимум: подумайте, почему вероятности от этого не меняются, — об этом спросят на защите.
2. Метки классов переведите в унитарный код (англ. *one-hot encoding*): строка с единицей в столбце своего класса; добавьте к матрицам признаков столбец единиц.
3. Начните с нулевой матрицы параметров. В каждой эпохе посчитайте вероятности и обе потери, запомните параметры, если валидационная потеря стала наименьшей, и сделайте шаг спуска по градиенту уравнения 4.23 сразу для всех классов — одним матричным произведением.
4. Остановитесь, если валидационная потеря не улучшалась `patience` эпох подряд или эпохи кончились. Выполните готовую ячейку с графиком после самопроверки.

`LogisticRegression` и другие модели scikit-learn для своей модели не используйте — только для сравнения.

**Что почитать:**

- лекция 05, слайды «Многопеременная логистическая регрессия», «Перекрёстная энтропия», «Раннее прекращение»; книга, стр. 185–187 и 193–197; лекция 04, слайды «Шаг градиентного спуска» и «Пакетный, стохастический и мини-пакетный»;
- [Softmax](https://ru.wikipedia.org/wiki/Softmax), [Перекрёстная энтропия](https://ru.wikipedia.org/wiki/Перекрёстная_энтропия), [Ранняя остановка](https://ru.wikipedia.org/wiki/Ранняя_остановка) и [Унитарный код](https://ru.wikipedia.org/wiki/Унитарный_код) — Википедия;
- документация NumPy: [`np.eye`](https://numpy.org/doc/1.24/reference/generated/numpy.eye.html), [`np.argmax`](https://numpy.org/doc/1.24/reference/generated/numpy.argmax.html), [`ndarray.copy`](https://numpy.org/doc/1.24/reference/generated/numpy.ndarray.copy.html), [транслирование (broadcasting)](https://numpy.org/doc/1.24/user/basics.broadcasting.html);
- документация scikit-learn: [`accuracy_score`](https://scikit-learn.org/1.3/modules/generated/sklearn.metrics.accuracy_score.html), [`LogisticRegression`](https://scikit-learn.org/1.3/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [ ]:
def softmax(scores):
    """Переводит оценки классов в вероятности функцией softmax.

    Args:
        scores: Матрица оценок формы (m, K): строка — образец, столбец — класс.

    Returns:
        Матрица вероятностей той же формы; сумма каждой строки равна 1.
    """
    ...  # Ваш код


def cross_entropy(proba, targets_onehot):
    """Считает перекрёстную энтропию.

    Args:
        proba: Матрица вероятностей классов формы (m, K).
        targets_onehot: Унитарный код меток той же формы: 1 в столбце истинного класса.

    Returns:
        Средняя по образцам перекрёстная энтропия.
    """
    ...  # Ваш код

In [ ]:
toy_scores = np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
toy_proba = softmax(toy_scores)
assert np.shape(toy_proba) == (2, 3) and np.allclose(toy_proba.sum(axis=1), 1), 'Сумма вероятностей строки равна 1'
assert np.allclose(toy_proba[1], 1 / 3), 'При равных оценках классы равновероятны'
assert np.allclose(toy_proba[0], [0.0900, 0.2447, 0.6652], atol=1e-4), 'softmax([1, 2, 3]) ≈ [0,090; 0,245; 0,665]'
assert np.allclose(softmax(toy_scores + 1000), toy_proba), 'Большие оценки переполняют exp: сдвиньте оценки строки'
toy_onehot = np.array([[0, 0, 1], [1, 0, 0]])
assert np.isclose(cross_entropy(toy_proba, toy_onehot), 0.7531, atol=1e-3), 'Перекрёстная энтропия примера ≈ 0,753'

'Задание 4, функции: самопроверка пройдена'

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

learning_rate = 0.5
max_epochs = 5000
patience = 1000  # Сколько эпох подряд валидационной потере разрешено не улучшаться

train_losses, val_losses = ..., ...  # Ваш код
best_epoch, theta_best = ..., ...  # Ваш код
val_accuracy_own, val_accuracy_sklearn = ..., ...  # Ваш код

best_epoch, len(val_losses), val_accuracy_own, val_accuracy_sklearn

In [ ]:
assert np.shape(theta_best) == (5, 3), 'theta_best — матрица (4 признака + свободный член) × 3 класса'
assert len(train_losses) == len(val_losses), 'Потери на обоих наборах считаются в каждой эпохе'
assert np.isclose(train_losses[0], np.log(3)), 'До первого шага с нулевыми параметрами потеря равна log 3'
assert len(val_losses) < max_epochs, 'Ранняя остановка должна прервать обучение раньше max_epochs'
assert np.isclose(val_losses[best_epoch], min(val_losses)), 'best_epoch — эпоха минимума валидационной потери'
assert len(val_losses) - best_epoch >= patience, 'После минимума нужно подождать patience эпох, а не остановиться сразу'
assert train_losses[-1] < train_losses[best_epoch], 'После лучшей эпохи обучающая потеря продолжает падать'
assert val_accuracy_own >= 0.9, 'Доля правильных ответов softmax-регрессии на валидации должна быть не ниже 0,9'
assert abs(val_accuracy_own - val_accuracy_sklearn) <= 0.07, 'Доля правильных ответов далека от LogisticRegression'

'Задание 4: самопроверка пройдена'

In [ ]:
fig, axis = plt.subplots(figsize=(9, 4))
axis.plot(train_losses, label='обучающий набор')
axis.plot(val_losses, label='валидационный набор')
axis.axvline(best_epoch, color='red', linestyle='--', label=f'лучшая эпоха {best_epoch}')
axis.set_ylim(0.75 * min(val_losses), 1.5 * min(val_losses))  # Начало кривых (около log 3) обрезано: виден минимум
axis.set_xlabel('Эпоха')
axis.set_ylabel('Перекрёстная энтропия')
axis.legend()
axis.grid(alpha=0.3)
plt.show()

По своему графику: (1) что происходит с обучающей и валидационной потерей после лучшей эпохи и почему валидационная растёт? (2) Как может расти валидационная перекрёстная энтропия, если доля правильных ответов при этом почти не меняется?

*Ваш ответ:* …

## Готово к защите

Перед тем как звать преподавателя, проверьте:

- ядро перезапущено (Kernel → Restart), подготовка данных и защищаемые задания выполнены сверху вниз без ошибок;
- самопроверки защищаемых заданий напечатали «самопроверка пройдена», графики построены;
- ответы в полях «*Ваш ответ:*» написаны своими словами;
- вы можете объяснить любую строку своего кода и знаете, где поменять скорость обучения, степень полинома, `alpha`, `C` или `patience`.